# Testando versão original para traçar baseline

In [48]:
from pathlib import Path
import os
import sys

print("cwd:", Path.cwd())
print("sys.path[0]:", sys.path[0])
print("primeiros caminhos:")
for p in sys.path[:5]:
    print("  ", p)

cwd: c:\Users\renna\OneDrive\Área de Trabalho\Projeto\FAPESP\FEMa-bases-investigation\baseline
sys.path[0]: C:\Users\renna\OneDrive\Área de Trabalho\Projeto\FAPESP\FEMa-bases-investigation\baseline
primeiros caminhos:
   C:\Users\renna\OneDrive\Área de Trabalho\Projeto\FAPESP\FEMa-bases-investigation\baseline
   c:\Users\renna\OneDrive\Área de Trabalho\Projeto\FAPESP\FEMa-bases-investigation
   C:\
   C:\Users\renna\AppData\Local\Programs\Python\Python313\python313.zip
   C:\Users\renna\AppData\Local\Programs\Python\Python313\DLLs


In [49]:
from pathlib import Path
import sys

root = Path.cwd().parent

if str(root) not in sys.path:
    sys.path.insert(0, str(root))

print(sys.path[:5])

['C:\\Users\\renna\\OneDrive\\Área de Trabalho\\Projeto\\FAPESP\\FEMa-bases-investigation\\baseline', 'c:\\Users\\renna\\OneDrive\\Área de Trabalho\\Projeto\\FAPESP\\FEMa-bases-investigation', 'C:\\', 'C:\\Users\\renna\\AppData\\Local\\Programs\\Python\\Python313\\python313.zip', 'C:\\Users\\renna\\AppData\\Local\\Programs\\Python\\Python313\\DLLs']


In [50]:
import datasets
print(datasets)
print(datasets.__file__)

<module 'datasets' from 'c:\\Users\\renna\\OneDrive\\Área de Trabalho\\Projeto\\FAPESP\\FEMa-bases-investigation\\datasets\\__init__.py'>
c:\Users\renna\OneDrive\Área de Trabalho\Projeto\FAPESP\FEMa-bases-investigation\datasets\__init__.py


In [51]:
from datasets import load_dataset, LoadedData

print(load_dataset)
print(LoadedData)

<function load_dataset at 0x0000021F07DCEDE0>
<class 'datasets.loader.LoadedData'>


In [52]:
from pathlib import Path

root = Path.cwd().parent

print(root)
print(root.exists())
print((root / "datasets").exists())
print((root / "datasets" / "__init__.py").exists())

c:\Users\renna\OneDrive\Área de Trabalho\Projeto\FAPESP\FEMa-bases-investigation
True
True
True


In [53]:
import sys
print(sys.executable)

c:\Users\renna\OneDrive\Área de Trabalho\Projeto\FAPESP\FEMa-bases-investigation\.venv\Scripts\python.exe


In [54]:
import sys
from pathlib import Path

root = Path.cwd().resolve()
while root != root.parent and not (root / "datasets.py").exists():
    root = root.parent

if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from datasets import load_dataset, LoadedData

data = load_dataset(dataset_name="fetal_health")

X_train, y_train = data.X_train, data.y_train
X_test, y_test = data.X_test, data.y_test
X_val, y_val = data.X_val, data.y_val

In [55]:
import numpy as np
# Junta treino + validação
X_train = np.concatenate([X_train, X_val], axis=0)
y_train = np.concatenate([y_train, y_val], axis=0)

print("Novo conjunto de treino:", X_train.shape, y_train.shape)
print("Conjunto de teste:", X_test.shape, y_test.shape)

Novo conjunto de treino: (1807, 21) (1807,)
Conjunto de teste: (319, 21) (319,)


In [56]:
from fem_basis import Basis
from fema_classifier  import FEMaClassifier
from fema_regression  import FEMaRegressor

In [57]:
model = FEMaClassifier(basis=Basis.shepardBasis)
print(y_train.shape)
print(type(y_train))

(1807,)
<class 'numpy.ndarray'>


In [58]:
y_train = y_train.reshape(-1, 1)
y_test = y_test.reshape(-1, 1)

In [59]:
model.fit(X_train, y_train)

In [60]:
print(X_train.shape)
print(y_train.shape)
print(X_test.shape)
print(y_test.shape)

(1807, 21)
(1807, 1)
(319, 21)
(319, 1)


In [61]:
# Parâmetro da base
z = 2  # ajuste para o mesmo valor utilizado no código original

# Predição
y_pred, confidence = model.predict(X_test, z)

# Armazena para comparação
results = {
    "y_true": y_test.ravel(),
    "y_pred": y_pred.astype(int),
    "confidence": confidence,
}

print(f"Número de amostras: {len(results['y_true'])}")
print("Primeiras predições:")
print(results["y_pred"][:10])

print("\nPrimeiras classes reais:")
print(results["y_true"][:10])

Número de amostras: 319
Primeiras predições:
[2 1 1 1 1 1 1 1 1 2]

Primeiras classes reais:
[1 1 1 1 1 1 1 1 1 1]


In [62]:
from metrics.registry import compute_all, metrics_for_context

# Calcula todas as métricas de classificação
results = compute_all(
    metric_names=metrics_for_context("classifier"),
    y_true=y_test.ravel(),
    y_pred=y_pred.astype(int),
    y_score=confidence,
    n_classes=len(np.unique(y_train))
)

print("Resultados:")
for metric, value in results.items():
    if value is None:
        print(f"{metric:20s}: None")
    else:
        print(f"{metric:20s}: {value:.6f}")

Resultados:
accuracy            : 0.833856
balanced_accuracy   : 0.538245
precision           : 0.395301
recall              : 0.403684
f1                  : 0.399448
roc_auc             : None
mcc                 : 0.552707


c:\Users\renna\OneDrive\Área de Trabalho\Projeto\FAPESP\FEMa-bases-investigation\.venv\Lib\site-packages\sklearn\metrics\_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


In [63]:
new_model = FEMaClassifier(k=27, basis=Basis.shepardBasis)
new_model.fit(X_train,y_train)

z = 3.34
y_pred, confidence = model.predict(X_test, z)

# Armazena para comparação
results2 = {
    "y_true": y_test.ravel(),
    "y_pred": y_pred.astype(int),
    "confidence": confidence,
}

results2 = compute_all(
    metric_names=metrics_for_context("classifier"),
    y_true=y_test.ravel(),
    y_pred=y_pred.astype(int),
    y_score=confidence,
    n_classes=len(np.unique(y_train))
)

print("Resultados:")
for metric, value in results2.items():
    if value is None:
        print(f"{metric:20s}: None")
    else:
        print(f"{metric:20s}: {value:.6f}")


Resultados:
accuracy            : 0.833856
balanced_accuracy   : 0.538245
precision           : 0.395301
recall              : 0.403684
f1                  : 0.399448
roc_auc             : None
mcc                 : 0.552707


c:\Users\renna\OneDrive\Área de Trabalho\Projeto\FAPESP\FEMa-bases-investigation\.venv\Lib\site-packages\sklearn\metrics\_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


"accuracy": 0.8432601880877743,
"balanced_accuracy": 0.5235826001955034,
"precision": 0.376477233229058,
"recall": 0.3926869501466276,
"f1": 0.3839214113873296,
"roc_auc": null,
"mcc": 0.523198444221588

In [64]:
new_model = FEMaClassifier(k=27, basis=Basis.radialBasis)
new_model.fit(X_train,y_train)

z = 0.5068077717443549

y_pred, confidence = model.predict(X_test, z)

# Armazena para comparação
results2 = {
    "y_true": y_test.ravel(),
    "y_pred": y_pred.astype(int),
    "confidence": confidence,
}

results2 = compute_all(
    metric_names=metrics_for_context("classifier"),
    y_true=y_test.ravel(),
    y_pred=y_pred.astype(int),
    y_score=confidence,
    n_classes=len(np.unique(y_train))
)

print("Resultados:")
for metric, value in results2.items():
    if value is None:
        print(f"{metric:20s}: None")
    else:
        print(f"{metric:20s}: {value:.6f}")


Resultados:
accuracy            : 0.833856
balanced_accuracy   : 0.538245
precision           : 0.395301
recall              : 0.403684
f1                  : 0.399448
roc_auc             : None
mcc                 : 0.552707


c:\Users\renna\OneDrive\Área de Trabalho\Projeto\FAPESP\FEMa-bases-investigation\.venv\Lib\site-packages\sklearn\metrics\_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


"n_train_total": 1488,
  "n_test": 319,
  "metrics": {
    "accuracy": 0.8338557993730408,
    "balanced_accuracy": 0.5320136852394918,
    "precision": 0.3657035848047084,
    "recall": 0.3990102639296188,
    "f1": 0.3816055955736507,
    "roc_auc": null,
    "mcc": 0.5074612485365817
  },